# 1. Import Thư Viện Cần Thiết
Cài đặt Ultralytics cho YOLO11m, Supabase để lấy Data, và HuggingFace để tải/chép đè Model.

In [ ]:
!pip install -q supabase huggingface_hub ultralytics
import os, requests, yaml
from supabase import create_client
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

# 2. Lấy Chìa Khóa Bảo Mật (Kaggle Secrets)
Vào Add-ons -> Secrets trên Kaggle để điền các mã biến này.

In [ ]:
user_secrets = UserSecretsClient()
SUPABASE_URL = user_secrets.get_secret("SUPABASE_URL")
SUPABASE_KEY = user_secrets.get_secret("SUPABASE_KEY")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# 3. Tự Động Kéo Dữ Liệu Tách Từ Supabase & Lập Dataset mới
Lấy 10 class gốc của model YOLO11m, sau đó nạp thêm các hình con vật mới.

In [ ]:
dataset_dir = "/kaggle/working/datasets"
os.makedirs(f"{dataset_dir}/images/train", exist_ok=True)
os.makedirs(f"{dataset_dir}/labels/train", exist_ok=True)

# 10 Loài gốc bạn đã train
base_classes = ['ant', 'butterfly', 'cockroach', 'dragonfly', 'fly', 'grasshopper', 'honeybee', 'ladybug', 'mosquito', 'spider']

# Kéo các ảnh mới từ Collections đã được AI dán nhãn YOLO
response = supabase.table("collections").select("*").neq("ai_yolo_label", "null").execute()
data = response.data

# Nối thêm class mới nếu AI phát hiện loài ngoài 10 con trên
class_names = base_classes.copy()
for item in data:
    if item['insect_id'] not in class_names and item['insect_id'] != 'unknown_insect':
        class_names.append(item['insect_id'])

# Ghi file data.yaml
yaml_data = {
    'path': dataset_dir,
    'train': "images/train",
    'val': "images/train",
    'nc': len(class_names),
    'names': {i: name for i, name in enumerate(class_names)}
}
with open(f"{dataset_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_data, f)

# Tải ảnh và ghi file txt bbox YOLO
for idx, obj in enumerate(data):
    if not obj['photo_path'] or not obj['ai_yolo_label'] or obj['insect_id'] == 'unknown_insect': 
        continue
    
    try:
        img_data = requests.get(obj['photo_path']).content
        img_name = f"new_insect_{idx}.jpg"
        with open(f"{dataset_dir}/images/train/{img_name}", 'wb') as handler:
            handler.write(img_data)
            
        # Tính ID dựa trên list gộp class_names
        class_id = class_names.index(obj['insect_id'])
        # Lấy Bbox từ db (Bỏ mã 0 ảo ban đầu AI sinh)
        raw_bbox = obj['ai_yolo_label'].split(" ", 1)[1] 
        yolo_line = f"{class_id} {raw_bbox}"
        
        with open(f"{dataset_dir}/labels/train/new_insect_{idx}.txt", 'w') as f:
            f.write(yolo_line)
    except Exception as e:
        print(f"Lỗi tải ảnh {idx}: {e}")
        
print(f"Tải xưởng Data tự động hoàn tất! Tổng cộng {len(class_names)} loài được hỗ trợ.")

# 4. Tải Não Yolo11m Cũ & Bắt Đầu Học (Fine-tuning)
Học theo phương pháp Transfer Learning, Learning Rate nhỏ, Ảnh size 640.

In [ ]:
from huggingface_hub import hf_hub_download

try:
    # Điền ID Kho chứa Hugging Face của bạn. Vd: nguyenviet2709/insect_yolo11m
    repo_huggingface = "Tên_TK/Tên_Repo" 
    model_path = hf_hub_download(repo_id=repo_huggingface, filename="best.pt", token=HF_TOKEN)
    print("Đã tải mô hình cũ từ Hugging Face!")
except:
    print("Hệ thống không tìm thấy mô hình cũ, tiến hành tải YOLO11m chuẩn của Ultralytics!")
    model_path = "yolo11m.pt"

model = YOLO(model_path)

# Fine-tune: Dùng augment mạnh (mosaic), lr thấp (lr0=0.001) để không hỏng trọng số 10 loài đã train
results = model.train(
    data=f"{dataset_dir}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    mosaic=1.0,
    device=0
)

# 5. Tự Động Push Model Mới Về Đám Mây Hugging Face

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

try:
    api.upload_file(
        path_or_fileobj="/kaggle/working/runs/detect/train/weights/best.pt",
        path_in_repo="best.pt",
        repo_id=repo_huggingface,
        repo_type="model",
        token=HF_TOKEN,
        commit_message="Auto-finetune with new mysterious insects!"
    )
    print("🎉 Cập nhật Não AI thành công! Hệ thống sẵn sàng cho lần quét Mobile tiếp theo.")
except Exception as e:
    print(f"Lỗi Upload lên HuggingFace: {e}")